# Generative Models — VAE and DCGAN

**Author:** Shivani Bokka  
**Dataset:** MNIST (torchvision)  
**Goal:** Build Variational Autoencoders and Deep Convolutional GANs from scratch on MNIST

---

## What Is This Notebook About?

This notebook is a **complete guide to generative models** — neural networks that can **create new data** rather than just classify or regress on existing data. We cover two major families:

1. **Variational Autoencoders (VAEs)** — probabilistic latent variable models that learn a structured latent space
2. **Deep Convolutional GANs (DCGANs)** — adversarial games between a generator and discriminator

By the end, you'll be able to generate realistic handwritten digits, interpolate between digits in latent space, and understand why GANs are notoriously unstable to train.

---

## Why Generative Models Matter

| Application | Generative Model Used |
|-------------|----------------------|
| AI image generation (DALL-E, Midjourney) | Diffusion models (successor to VAE/GAN) |
| Deepfakes | GANs |
| Drug discovery | VAE over molecular graphs |
| Data augmentation | GANs and VAEs |
| Anomaly detection | VAE reconstruction error |
| Text generation | Autoregressive models (GPT) |

---

## Step 1 — Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import linalg
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # scale to [-1, 1]
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(mnist_test, batch_size=256, shuffle=False, num_workers=0)

print(f'Training samples: {len(mnist_train)}')
print(f'Test samples: {len(mnist_test)}')
print('MNIST loaded successfully!')

---

## Section 2 — Generative vs Discriminative Models

### What Is the Difference?

| Property | Discriminative Model | Generative Model |
|----------|---------------------|------------------|
| **Learns** | P(y \| x) — the label given data | P(x) or P(x \| y) — the data distribution |
| **Can classify** | Yes (that's all it does) | Yes (via Bayes' theorem) |
| **Can generate new samples** | No | Yes |
| **Examples** | Logistic Regression, CNNs, SVMs | VAE, GAN, Diffusion Models |
| **Question answered** | "Given this image, which class?" | "What does class X look like?" |
| **Training data needed** | Labeled | Can be unsupervised |

### Analogy

> **Discriminative:** A wine expert who can taste a wine and name the vineyard (classification).  
> **Generative:** A master winemaker who can create a wine that tastes exactly like a specific vineyard (generation).

The generative task is harder — you have to understand the data distribution deeply, not just draw boundaries between classes.

---

---

## Section 3 — Variational Autoencoder: Theory

### The Problem with Standard Autoencoders

A regular autoencoder learns to compress data to a latent code and decompress it back. But the latent space is **unstructured** — you can't sample from it to generate new images because empty regions of latent space produce garbage.

### The VAE Solution: Learn a Distribution

Instead of mapping input x to a point z, the encoder maps x to a **distribution** over z:

```
Encoder: x → (μ, log σ²)    ← mean and log-variance of q(z|x)
Reparameterize: z = μ + σ · ε    where ε ~ N(0, I)
Decoder: z → x_reconstructed
```

### The ELBO Loss

The VAE optimizes the **Evidence Lower BOund (ELBO)**:

```
ELBO = E[log p(x|z)] - KL(q(z|x) || p(z))
     = Reconstruction Loss - KL Divergence
```

- **Reconstruction Loss** `E[log p(x|z)]`: How well does the decoder reconstruct x from z? (Binary Cross-Entropy for MNIST pixels in [0,1])
- **KL Divergence** `KL(q(z|x) || p(z))`: How close is the learned posterior to the standard normal N(0,I)? This regularizes the latent space.

For Gaussian encoder with diagonal covariance, KL has a closed form:

```
KL = -0.5 × Σ (1 + log σ_i² - μ_i² - σ_i²)
```

### The Reparameterization Trick

Why not just sample z ~ N(μ, σ²) directly? Because we can't backpropagate through a random sample.

The trick: instead of sampling z directly, sample ε ~ N(0,I) and compute z = μ + σ·ε. Now:
- The randomness (ε) is separate from the learned parameters
- Gradients can flow back through μ and σ

```
WITHOUT reparameterization:  z ←── sample ←── N(μ, σ²)   ← no gradient!
WITH reparameterization:     z = μ + σ · ε              ← gradient flows through μ, σ
                              where ε ~ N(0, I)          ← separate random node
```

### VAE Architecture (ASCII)

```
Input x (1×28×28)
     │
  ┌──┴────────────────────────────────┐
  │         ENCODER                  │
  │  Conv(1→32,4,2) → ReLU          │
  │  Conv(32→64,4,2) → ReLU         │
  │  Flatten → Linear → (μ, log σ²) │
  └──────────────────────────────────┘
          │            │
          μ          log σ²
          │            │
          └──── z = μ + σ·ε  ←── ε ~ N(0,I)   [REPARAMETERIZE]
                    │
  ┌─────────────────┴────────────────────┐
  │             DECODER                 │
  │  Linear → Unflatten                 │
  │  ConvTranspose(64→32,4,2) → ReLU   │
  │  ConvTranspose(32→1,4,2) → Tanh    │
  └──────────────────────────────────────┘
          │
   Reconstructed x̂ (1×28×28)
```

---

---

## Section 4 — Building and Training the VAE

In [ ]:
LATENT_DIM = 16

class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # 28→14
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 14→7
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        h = self.conv(x).view(x.size(0), -1)  # flatten
        return self.fc_mu(h), self.fc_logvar(h)


class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 7→14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),   # 14→28
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 64, 7, 7)
        return self.deconv(h)


class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + std * eps
        return mu  # at inference, use mean

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """ELBO loss = Reconstruction + beta * KL Divergence."""
    # Reconstruction: MSE between images in [-1, 1]
    recon_loss = F.mse_loss(recon_x, x, reduction='sum') / x.size(0)
    # KL Divergence: -0.5 * sum(1 + log_var - mu^2 - var)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


# Training loop
torch.manual_seed(42)
vae = VAE().to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)
VAE_EPOCHS = 20

recon_losses, kl_losses, total_losses = [], [], []

print('Training VAE for 20 epochs...')
for epoch in range(VAE_EPOCHS):
    vae.train()
    epoch_recon, epoch_kl, epoch_total = 0.0, 0.0, 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        vae_optimizer.zero_grad()
        recon, mu, logvar = vae(imgs)
        loss, r_loss, k_loss = vae_loss(recon, imgs, mu, logvar)
        loss.backward()
        vae_optimizer.step()
        epoch_total += loss.item()
        epoch_recon += r_loss.item()
        epoch_kl += k_loss.item()
    n = len(train_loader)
    total_losses.append(epoch_total / n)
    recon_losses.append(epoch_recon / n)
    kl_losses.append(epoch_kl / n)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:2d}/{VAE_EPOCHS} | '
              f'Total: {total_losses[-1]:.2f} | '
              f'Recon: {recon_losses[-1]:.2f} | '
              f'KL: {kl_losses[-1]:.2f}')

print('VAE training complete!')

# Plot reconstruction and KL losses separately
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = list(range(1, VAE_EPOCHS + 1))

ax1.plot(epochs_range, recon_losses, color='steelblue', marker='o', markersize=3)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Reconstruction Loss (MSE per sample)')
ax1.set_title('VAE Reconstruction Loss per Epoch')

ax2.plot(epochs_range, kl_losses, color='tomato', marker='o', markersize=3)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('KL Divergence per sample')
ax2.set_title('VAE KL Divergence per Epoch')

plt.tight_layout()
plt.show()

### How to Read This Chart: VAE Loss Components

Two side-by-side line charts showing **reconstruction loss** (left) and **KL divergence** (right) over 20 training epochs.

**Reconstruction Loss (left):**
- Measures how well the decoder reconstructs the input image from the latent code z.
- **MSE (Mean Squared Error)** per sample — average pixel-level squared difference between input and reconstruction.
- **Should decrease** over epochs as the decoder gets better at reconstructing digits.
- If it plateaus early, try a deeper or wider decoder.

**KL Divergence (right):**
- Measures how far the learned posterior q(z|x) = N(μ, σ²) is from the prior p(z) = N(0, I).
- **KL = 0** would mean the encoder always outputs exactly N(0,1) — it has "forgotten" the input.
- **KL rising then stabilizing** is healthy — the model is learning to use the latent space efficiently without collapsing to the prior.
- If KL = 0 throughout, you have **posterior collapse** — the decoder ignores z.

> **The tension:** Minimizing reconstruction loss wants μ and σ to encode as much information as possible (large deviation from prior). KL minimization wants to push back toward N(0,I). The beta hyperparameter controls this trade-off.

---

## Section 5 — Visualizing VAE Reconstructions

In [ ]:
# Show 8 original vs reconstructed MNIST images side by side
vae.eval()
test_imgs, test_labels = next(iter(test_loader))
test_imgs = test_imgs[:8].to(device)

with torch.no_grad():
    recon_imgs, _, _ = vae(test_imgs)

# Display
def unnorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(unnorm(test_imgs[i]).cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', loc='left', fontsize=10, pad=2)

    axes[1, i].imshow(unnorm(recon_imgs[i]).cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', loc='left', fontsize=10, pad=2)

plt.suptitle('VAE Reconstruction Quality: Original (top) vs Reconstructed (bottom)', y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: VAE Reconstruction Quality

Two rows of 8 MNIST digit images:

- **Top row:** Original test images — these are real handwritten digits from MNIST that the VAE has never seen during training.
- **Bottom row:** VAE reconstructions — what the model produces when it encodes these images to latent vectors and then decodes them.

**What to look for:**
- **Sharpness:** Are the reconstructions clean and sharp, or blurry? VAEs tend to produce slightly blurry reconstructions because they average over the posterior distribution.
- **Accuracy:** Does each reconstruction show the same digit as the original? If the model confuses a 9 for a 4, the latent space hasn't learned well.
- **Style preservation:** Does the reconstruction look like a digit of similar style — thick/thin strokes, slant?

> **Why are VAE reconstructions slightly blurry?** The decoder must reconstruct x from a sampled z, not the true z. Averaging over the distribution naturally blurs sharp features. GANs produce sharper images but at the cost of training instability.

---

## Section 6 — Visualizing the VAE Latent Space

In [ ]:
# Encode 2000 test images → extract mu vectors → PCA to 2D → scatter colored by digit class
vae.eval()
all_mu, all_labels = [], []
n_collected = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        mu, _ = vae.encoder(imgs)
        all_mu.append(mu.cpu().numpy())
        all_labels.append(labels.numpy())
        n_collected += len(labels)
        if n_collected >= 2000:
            break

all_mu = np.concatenate(all_mu, axis=0)[:2000]
all_labels = np.concatenate(all_labels, axis=0)[:2000]

# PCA to 2D
pca = PCA(n_components=2)
mu_2d = pca.fit_transform(all_mu)
variance_explained = pca.explained_variance_ratio_
print(f'PCA Variance explained: PC1={variance_explained[0]:.1%}, PC2={variance_explained[1]:.1%}')

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(mu_2d[:, 0], mu_2d[:, 1], c=all_labels,
                     cmap='tab10', alpha=0.6, s=10)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Digit Class', fontsize=10)
cbar.set_ticks(list(range(10)))
ax.set_xlabel(f'PC1 ({variance_explained[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({variance_explained[1]:.1%} variance)')
ax.set_title('VAE Latent Space: 2000 Test Images projected to 2D via PCA')
plt.tight_layout()
plt.show()

### How to Read This Chart: VAE Latent Space

This scatter plot shows 2000 test digit images encoded into the 16-dimensional VAE latent space, then projected to 2D using PCA.

- **Each dot** is one MNIST test image.
- **Position** reflects the latent code μ (encoded by the VAE encoder).
- **Color** indicates the true digit class (0-9, shown in the colorbar).
- The **axes (PC1, PC2)** are the two principal components — linear combinations of the 16 latent dimensions that explain the most variance.

**What to look for:**
- **Clustered colors** = the VAE has learned to separate digit classes in latent space — similar digits are near each other.
- **Overlapping regions** = digits that look similar (e.g., 3 and 8, 4 and 9) tend to overlap in latent space.
- **Smooth transitions between clusters** = the latent space is continuous — you can interpolate between digit classes.
- **Compact, well-separated clusters** indicate good disentanglement.

> **Key property of VAEs:** The KL term forces the latent space toward N(0,I), which means the clusters should be arranged near the origin and roughly spherical. This makes random sampling (z ~ N(0,I)) likely to produce recognizable digits.

---

## Section 7 — Latent Space Interpolation

In [ ]:
# Interpolate between digit 3 and digit 7
vae.eval()

# Find one example of each digit in test set
test_imgs_all, test_labels_all = [], []
for imgs, labels in test_loader:
    test_imgs_all.append(imgs)
    test_labels_all.append(labels)
test_imgs_all = torch.cat(test_imgs_all)
test_labels_all = torch.cat(test_labels_all)

idx_3 = (test_labels_all == 3).nonzero(as_tuple=True)[0][0]
idx_7 = (test_labels_all == 7).nonzero(as_tuple=True)[0][0]

img_3 = test_imgs_all[idx_3:idx_3+1].to(device)
img_7 = test_imgs_all[idx_7:idx_7+1].to(device)

with torch.no_grad():
    mu_3, _ = vae.encoder(img_3)
    mu_7, _ = vae.encoder(img_7)

# Interpolate 6 steps between z_3 and z_7
N_INTERP = 8  # includes endpoints
alphas = torch.linspace(0, 1, N_INTERP)
interpolated = []
for alpha in alphas:
    z_interp = (1 - alpha) * mu_3 + alpha * mu_7
    with torch.no_grad():
        img_decoded = vae.decoder(z_interp)
    interpolated.append(img_decoded.cpu())

fig, axes = plt.subplots(1, N_INTERP, figsize=(2 * N_INTERP, 2.5))
for i, img in enumerate(interpolated):
    axes[i].imshow(unnorm(img).squeeze(), cmap='gray')
    axes[i].axis('off')
    alpha_val = alphas[i].item()
    if i == 0:
        axes[i].set_title('"3"', fontsize=10)
    elif i == N_INTERP - 1:
        axes[i].set_title('"7"', fontsize=10)
    else:
        axes[i].set_title(f'α={alpha_val:.2f}', fontsize=8)

plt.suptitle('VAE Latent Space Interpolation: Digit 3 → Digit 7', fontsize=12)
plt.tight_layout()
plt.show()

### How to Read This Chart: Latent Space Interpolation

This row of images shows a smooth morphing sequence from digit 3 to digit 7, created by **linearly interpolating** between their latent codes.

- **Leftmost image:** Decoded from z_3 (the VAE encoding of a real "3") — should look like a clear digit 3.
- **Rightmost image:** Decoded from z_7 — should look like a clear digit 7.
- **Middle images:** Decoded from z = (1-α)·z_3 + α·z_7 for α = 0.14, 0.29, 0.43, 0.57, 0.71, 0.86.
- **α label** shows how far we are along the interpolation path (0=start, 1=end).

**What to look for:**
- **Smooth transition** — each intermediate image should be a plausible digit-like shape (not random noise). This is the signature of a well-structured latent space.
- **Gradual morphing** — the stroke gradually transforms from the 3's curved top to the 7's vertical stroke.
- **No discontinuities** — if there's a sudden jump between adjacent images, the latent space has gaps (poorly trained VAE).

> **This is impossible with a regular autoencoder.** Without the KL regularization forcing the latent space to be continuous, the midpoint z=(z_3+z_7)/2 falls in an empty region and decodes to garbage. The VAE's regularization ensures smooth coverage of latent space.

---

## Section 8 — Random Sampling from the VAE

In [ ]:
# Sample 100 random z ~ N(0, I), decode, display 10×10 grid
vae.eval()
with torch.no_grad():
    z_random = torch.randn(100, LATENT_DIM).to(device)
    gen_imgs = vae.decoder(z_random).cpu()

fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for i in range(100):
    ax = axes[i // 10][i % 10]
    ax.imshow(unnorm(gen_imgs[i]).squeeze(), cmap='gray')
    ax.axis('off')

plt.suptitle('VAE Random Generation: 100 Samples from z ~ N(0, I)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### How to Read This Chart: VAE Random Generation

This 10×10 grid shows 100 images generated by the VAE by sampling random latent codes from the standard normal distribution.

- **Each cell** is a generated image — not from the training set, not a reconstruction, but a fully **novel generation** from z ~ N(0, I).
- **No labels were used** — this is unconditional generation.

**What to look for:**
- **Recognizable digits:** Can you identify what digit each image represents? Good VAEs produce clear, readable digits.
- **Diversity:** Is there a mix of different digits (0-9) across the grid? If only a few digit types appear, the model has limited diversity.
- **Quality:** Are images sharp or blurry? VAEs typically produce slightly blurry results (compared to GANs).
- **Artifacts:** Are there any images that are just noise or uninterpretable? This happens when the KL term was too weak and some regions of N(0,1) map to empty latent space.

> **Practical note:** If you mostly see 0s and 1s with few other digits, the VAE's latent space is not well-balanced. Try increasing the beta (KL weight) to force more uniform coverage.

---

## Section 9 — DCGAN: Deep Convolutional GAN

### How a GAN Works: The Adversarial Game

A GAN trains two networks simultaneously in a minimax game:

```
  NOISE z ~ N(0, I)
       │
  ┌────┴──────────────────┐
  │       GENERATOR       │   ← Creates fake images from noise
  │  G(z) → fake image    │
  └────────────────────┬──┘
                       │
  Real images ──────→  │
                       ↓
  ┌────────────────────────┐
  │     DISCRIMINATOR      │   ← Classifies real vs fake
  │  D(x) → P(real)        │
  └────────────────────────┘
```

### The Minimax Loss

```
min_G max_D V(D, G) = E[log D(x)] + E[log(1 - D(G(z)))]

Discriminator maximizes:  D(x) → 1 for real,  D(G(z)) → 0 for fake
Generator minimizes:      D(G(z)) → 1  (fool the discriminator)
```

In practice, we use separate Binary Cross-Entropy losses:

```
D_loss = BCE(D(real), ones) + BCE(D(fake), zeros)
G_loss = BCE(D(G(z)), ones)   ← Generator wants D to say fake images are real
```

### DCGAN Architecture

DCGAN (Radford et al., 2015) stabilized GAN training with architectural guidelines:
- Use strided convolutions instead of pooling
- Use BatchNorm in both G and D
- Use ReLU in G (except output: Tanh)
- Use LeakyReLU in D (prevents dying gradients)
- No fully-connected layers

In [ ]:
Z_DIM = 100  # Noise dimension

class Generator(nn.Module):
    """DCGAN Generator: z (100) → 1×28×28 image."""
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            # Project and reshape: z → (128, 7, 7)
            nn.Linear(z_dim, 128 * 7 * 7),
            nn.BatchNorm1d(128 * 7 * 7),
            nn.ReLU(True),
            nn.Unflatten(1, (128, 7, 7)),
            # Upsample: 7 → 14
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # Upsample: 14 → 28
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """DCGAN Discriminator: 1×28×28 → scalar probability."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 28 → 14
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # 14 → 7
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


def weights_init(m):
    """DCGAN weight initialization."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

print('Generator and Discriminator defined.')
test_G = Generator()
test_D = Discriminator()
z_test = torch.randn(4, Z_DIM)
fake_test = test_G(z_test)
d_out_test = test_D(fake_test)
print(f'Generator output shape:     {fake_test.shape}  (4 fake images)')
print(f'Discriminator output shape: {d_out_test.shape}  (4 scalar probabilities)')

---

## Section 10 — Training the DCGAN

In [ ]:
# Full GAN training loop — 30 epochs
torch.manual_seed(42)

G = Generator().to(device)
D = Discriminator().to(device)
G.apply(weights_init)
D.apply(weights_init)

G_optimizer = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
D_optimizer = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion_gan = nn.BCELoss()

GAN_EPOCHS = 30
D_losses, G_losses, D_x_hist, D_G_z_hist = [], [], [], []

# Fixed noise for progress visualization
fixed_noise = torch.randn(25, Z_DIM, device=device)
# Store generated images at checkpoints
progress_images = {}

print('Training DCGAN for 30 epochs...')
for epoch in range(GAN_EPOCHS):
    epoch_d_loss, epoch_g_loss, epoch_dx, epoch_dgz = 0.0, 0.0, 0.0, 0.0
    n_batches = 0

    for real_imgs, _ in train_loader:
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        real_label = torch.ones(bs, 1, device=device) * 0.9  # label smoothing
        fake_label = torch.zeros(bs, 1, device=device)

        # ---- Train Discriminator ----
        D_optimizer.zero_grad()
        d_real = D(real_imgs)
        d_loss_real = criterion_gan(d_real, real_label)

        z = torch.randn(bs, Z_DIM, device=device)
        fake_imgs = G(z).detach()  # detach so G doesn't get gradient here
        d_fake = D(fake_imgs)
        d_loss_fake = criterion_gan(d_fake, fake_label)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        D_optimizer.step()

        # ---- Train Generator ----
        G_optimizer.zero_grad()
        z = torch.randn(bs, Z_DIM, device=device)
        fake_imgs = G(z)
        d_fake_for_G = D(fake_imgs)
        g_loss = criterion_gan(d_fake_for_G, torch.ones(bs, 1, device=device))
        g_loss.backward()
        G_optimizer.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        epoch_dx += d_real.mean().item()
        epoch_dgz += d_fake_for_G.mean().item()
        n_batches += 1

    D_losses.append(epoch_d_loss / n_batches)
    G_losses.append(epoch_g_loss / n_batches)
    D_x_hist.append(epoch_dx / n_batches)
    D_G_z_hist.append(epoch_dgz / n_batches)

    # Save images at checkpoints
    if (epoch + 1) in [1, 5, 10, 20, 30]:
        G.eval()
        with torch.no_grad():
            progress_images[epoch + 1] = G(fixed_noise).cpu()
        G.train()

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:2d}/{GAN_EPOCHS} | '
              f'D_loss: {D_losses[-1]:.4f} | G_loss: {G_losses[-1]:.4f} | '
              f'D(x): {D_x_hist[-1]:.3f} | D(G(z)): {D_G_z_hist[-1]:.3f}')

print('DCGAN training complete!')

# Plot D_loss and G_loss
fig, ax = plt.subplots(figsize=(10, 4))
epochs_range_gan = list(range(1, GAN_EPOCHS + 1))
ax.plot(epochs_range_gan, D_losses, label='Discriminator Loss', color='tomato')
ax.plot(epochs_range_gan, G_losses, label='Generator Loss', color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('DCGAN Training Dynamics: Generator vs Discriminator Loss')
ax.legend()
plt.tight_layout()
plt.show()

### How to Read This Chart: GAN Training Dynamics

This line chart shows the Generator and Discriminator losses over 30 training epochs.

- **The x-axis** is the epoch.
- **The y-axis** is the Binary Cross-Entropy loss.
- **Red line (Discriminator):** BCE loss of D on real and fake images.
- **Blue line (Generator):** BCE loss of G when trying to fool D.

**What healthy GAN training looks like:**
- Both losses should **fluctuate and eventually stabilize** near log(2) ≈ 0.693.
- D_loss near 0.693 means D assigns ~50% probability to real and fake images — it can't reliably tell them apart.
- G_loss near 0.693 means G's fakes are indistinguishable from real images.
- **This is Nash Equilibrium** — neither player can improve unilaterally.

**Warning signs:**
- D_loss → 0: Discriminator is winning — it easily identifies fakes. G is failing.
- G_loss → 0: Generator is winning — but this usually means mode collapse (G produces only one type of image).
- Wild oscillations: Unstable training — try lower learning rate or gradient clipping.

> **D(x) and D(G(z)):** Track these alongside losses. D(x) should be high (D says real images are real). D(G(z)) should start low and rise toward 0.5 as G improves.

---

## Section 11 — Mode Collapse Demo

In [ ]:
# Train a 'bad GAN': no BatchNorm in Generator, high learning rate
class BadGenerator(nn.Module):
    """Generator WITHOUT BatchNorm — prone to mode collapse."""
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128 * 7 * 7),
            nn.ReLU(True),          # No BatchNorm
            nn.Unflatten(1, (128, 7, 7)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),          # No BatchNorm
            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

torch.manual_seed(42)
bad_G = BadGenerator().to(device)
bad_D = Discriminator().to(device)

# High learning rate = instability
bad_G_opt = optim.Adam(bad_G.parameters(), lr=0.01, betas=(0.5, 0.999))
bad_D_opt = optim.Adam(bad_D.parameters(), lr=0.01, betas=(0.5, 0.999))

print('Training bad GAN (no BatchNorm, LR=0.01) for 10 epochs...')
for epoch in range(10):
    for real_imgs, _ in train_loader:
        bs = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # Train D
        bad_D_opt.zero_grad()
        d_real = bad_D(real_imgs)
        d_loss_real = criterion_gan(d_real, torch.ones(bs, 1, device=device))
        z = torch.randn(bs, Z_DIM, device=device)
        fake = bad_G(z).detach()
        d_loss_fake = criterion_gan(bad_D(fake), torch.zeros(bs, 1, device=device))
        (d_loss_real + d_loss_fake).backward()
        bad_D_opt.step()

        # Train G
        bad_G_opt.zero_grad()
        z = torch.randn(bs, Z_DIM, device=device)
        fake = bad_G(z)
        g_loss = criterion_gan(bad_D(fake), torch.ones(bs, 1, device=device))
        g_loss.backward()
        bad_G_opt.step()

print('Bad GAN training complete!')

# Generate samples from both GANs
bad_G.eval(); G.eval()
with torch.no_grad():
    z_show = torch.randn(25, Z_DIM, device=device)
    bad_samples = bad_G(z_show).cpu()
    good_samples = G(z_show).cpu()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

# Bad GAN row (top 5)
for i in range(5):
    axes[0, i].imshow(unnorm(bad_samples[i]).squeeze(), cmap='gray')
    axes[0, i].axis('off')
axes[0, 0].set_title('Bad GAN (no BN, LR=0.01)', fontsize=10, loc='left')

# Good GAN row (top 5)
for i in range(5):
    axes[1, i].imshow(unnorm(good_samples[i]).squeeze(), cmap='gray')
    axes[1, i].axis('off')
axes[1, 0].set_title('Good GAN (with BN, LR=2e-4)', fontsize=10, loc='left')

plt.suptitle('Mode Collapse vs Healthy GAN: Generated Images', fontsize=12)
plt.tight_layout()
plt.show()

### How to Read This Chart: Mode Collapse vs Healthy GAN

Two rows of 5 generated images from two differently configured GANs.

- **Top row (Bad GAN):** Generator without BatchNorm, trained with LR=0.01.
- **Bottom row (Good GAN):** Standard DCGAN with BatchNorm, LR=2e-4.

**Signs of Mode Collapse (bad GAN):**
- All 5 images look nearly identical (same digit, same style).
- The Generator has converged on a single mode — the one that most reliably fools the Discriminator.
- Images may be blurry, noisy, or unrecognizable.

**Signs of Healthy GAN (good GAN):**
- Images show **variety** — different digit types and styles.
- Digits are recognizable and sharp.
- The Generator is sampling from many different modes of the MNIST distribution.

**Why does BatchNorm help?**
BatchNorm normalizes activations across the batch, preventing any single generator channel from dominating. This keeps training stable and encourages the Generator to explore the full data distribution rather than collapsing to a single easy-to-fool mode.

---

## Section 12 — GAN Training Progression

In [ ]:
# Show 5×5 grids at epochs 1, 5, 10, 20, 30
checkpoint_epochs = [1, 5, 10, 20, 30]
fig, big_axes = plt.subplots(1, len(checkpoint_epochs), figsize=(20, 5))

for col_idx, ep in enumerate(checkpoint_epochs):
    if ep not in progress_images:
        big_axes[col_idx].axis('off')
        big_axes[col_idx].set_title(f'Epoch {ep}\n(not saved)', fontsize=9)
        continue

    imgs = progress_images[ep][:25]  # 25 images for 5×5
    # Create a 5×5 mosaic
    grid = torchvision.utils.make_grid(unnorm(imgs), nrow=5, padding=2)
    grid_np = grid.permute(1, 2, 0).numpy()

    big_axes[col_idx].imshow(grid_np, cmap='gray')
    big_axes[col_idx].axis('off')
    big_axes[col_idx].set_title(f'Epoch {ep}', fontsize=11)

plt.suptitle('DCGAN Training Progression: Generated Images at Different Epochs',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### How to Read This Chart: GAN Training Progression

Five panels (one per checkpoint epoch), each showing a 5×5 grid of generated images using the **same fixed noise vectors** at different training stages.

- **Same noise vectors** are used at every checkpoint — so you're seeing how the same random seeds evolve as training progresses.
- **Epoch 1:** Images are nearly pure noise or vague blobs — the generator has barely started learning.
- **Epoch 5:** Structure begins to emerge — you can see some digit-like shapes, but they're blurry and distorted.
- **Epoch 10:** Recognizable digits appear — strokes are visible, but still rough.
- **Epoch 20:** Sharper, cleaner digits — the generator has converged on realistic digit styles.
- **Epoch 30:** Near-realistic digits — close to MNIST quality.

**What to look for:**
- The **same noise vector** should map to the same digit across epochs (once the generator stabilizes).
- **Quality improvement** should be visible epoch by epoch.
- **Diversity:** Are different images in the grid different digits, or are they all converging to the same thing (mode collapse)?

> **Tip:** GANs are often judged by how quickly they reach good quality and how diverse their outputs are. A GAN that generates all 1s quickly is not better than one that slowly learns to generate diverse 0-9 digits.

---

## Section 13 — WGAN: Wasserstein GAN

### Why Standard GANs Are Unstable

The minimax GAN loss has a fundamental problem: when the Discriminator is too good, it assigns exactly 0 probability to all fake images. The gradient for the Generator then becomes **zero** — no learning signal.

### The Wasserstein Distance

The **Wasserstein-1 distance** (Earth Mover's Distance) measures the minimum cost to transform one distribution into another. Unlike KL or JS divergence, it provides meaningful gradients even when the distributions have non-overlapping support.

```
W(P_r, P_g) = inf_{γ ~ Π(P_r, P_g)} E_{(x,y)~γ} [||x - y||]
```

### Three Key Changes in WGAN

| Change | Standard GAN | WGAN |
|--------|-------------|------|
| Discriminator output | Sigmoid → [0,1] probability | No sigmoid → raw scalar |
| Loss | BCE | Wasserstein loss (no log) |
| Critic constraint | None | Weight clipping to [-0.01, 0.01] |
| Critic steps per G step | 1 | 5 (train critic more) |

```
WGAN Critic loss: E[f(x_real)] - E[f(G(z))]
WGAN G loss:      -E[f(G(z))]
```

The critic tries to **maximize** the gap between real and fake scores. The generator tries to **minimize** (make the critic score fakes as high as possible).

In [ ]:
class WGANCritic(nn.Module):
    """WGAN Critic: no Sigmoid output (raw score)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1)  # No sigmoid!
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
WGAN_EPOCHS = 30
CRITIC_STEPS = 5
CLIP_VALUE = 0.01

wgan_G = Generator().to(device)
wgan_C = WGANCritic().to(device)
wgan_G.apply(weights_init)
wgan_C.apply(weights_init)

wgan_G_opt = optim.RMSprop(wgan_G.parameters(), lr=5e-5)
wgan_C_opt = optim.RMSprop(wgan_C.parameters(), lr=5e-5)

wgan_G_losses, wgan_C_losses = [], []

print('Training WGAN for 30 epochs...')
for epoch in range(WGAN_EPOCHS):
    epoch_c_loss, epoch_g_loss = 0.0, 0.0
    n_batches = 0
    data_iter = iter(train_loader)

    # Rough number of generator updates per epoch
    n_gen_updates = len(train_loader) // CRITIC_STEPS

    for _ in range(n_gen_updates):
        # ---- Train Critic 5 times ----
        for _ in range(CRITIC_STEPS):
            try:
                real_imgs, _ = next(data_iter)
            except StopIteration:
                data_iter = iter(train_loader)
                real_imgs, _ = next(data_iter)
            bs = real_imgs.size(0)
            real_imgs = real_imgs.to(device)
            wgan_C_opt.zero_grad()
            z = torch.randn(bs, Z_DIM, device=device)
            fake = wgan_G(z).detach()
            c_loss = -wgan_C(real_imgs).mean() + wgan_C(fake).mean()
            c_loss.backward()
            wgan_C_opt.step()
            # Weight clipping
            for p in wgan_C.parameters():
                p.data.clamp_(-CLIP_VALUE, CLIP_VALUE)
            epoch_c_loss += c_loss.item()

        # ---- Train Generator once ----
        wgan_G_opt.zero_grad()
        z = torch.randn(bs, Z_DIM, device=device)
        fake = wgan_G(z)
        g_loss = -wgan_C(fake).mean()
        g_loss.backward()
        wgan_G_opt.step()
        epoch_g_loss += g_loss.item()
        n_batches += 1

    if n_batches > 0:
        wgan_C_losses.append(epoch_c_loss / (n_batches * CRITIC_STEPS))
        wgan_G_losses.append(epoch_g_loss / n_batches)

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}/{WGAN_EPOCHS} | '
              f'Critic Loss: {wgan_C_losses[-1]:.4f} | '
              f'G Loss: {wgan_G_losses[-1]:.4f}')

print('WGAN training complete!')

# Compare DCGAN vs WGAN training stability
ep_range_dcgan = list(range(1, len(D_losses) + 1))
ep_range_wgan = list(range(1, len(wgan_C_losses) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep_range_dcgan, D_losses, color='tomato', label='DCGAN Discriminator Loss')
ax1.plot(ep_range_dcgan, G_losses, color='steelblue', label='DCGAN Generator Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('DCGAN Training Losses')
ax1.legend()

ax2.plot(ep_range_wgan, wgan_C_losses, color='darkorange', label='WGAN Critic Loss')
ax2.plot(ep_range_wgan, wgan_G_losses, color='teal', label='WGAN Generator Loss')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Wasserstein Loss')
ax2.set_title('WGAN Training Losses')
ax2.legend()

plt.suptitle('DCGAN vs WGAN Training Stability Comparison', fontsize=13)
plt.tight_layout()
plt.show()

### How to Read This Chart: DCGAN vs WGAN Stability

Two side-by-side panels comparing loss curves for DCGAN (left) and WGAN (right) over 30 epochs.

**DCGAN (left):**
- D_loss and G_loss should oscillate and ideally converge near 0.693 (log 2).
- Common pattern: D_loss drops first (discriminator wins), then G_loss drops as the generator catches up.
- High variance / oscillations are normal for standard GANs.

**WGAN (right):**
- The Wasserstein loss is **not** bounded to [0,1] — it can be any real number.
- **The Critic loss (negative Wasserstein distance)** should increase toward 0 as training progresses (the two distributions become more similar).
- **Much smoother curves** — this is the key advantage of WGAN: the loss is a meaningful metric of training progress.
- In DCGAN, you can't tell from the loss whether training is going well. In WGAN, a decreasing Critic loss reliably indicates improving generation quality.

> **Key WGAN advantage:** The Wasserstein loss serves as a reliable proxy for image quality — lower Critic loss correlates with better generated images. This makes WGAN much easier to tune and debug.

---

## Section 14 — Conditional VAE

### What Is a Conditional VAE?

A standard VAE generates digits randomly — we have no control over which digit is generated. A **Conditional VAE (CVAE)** takes a class label as additional input, learning the conditional distribution P(x|y).

We concatenate the **one-hot label vector** to:
1. The encoder input (before convolution, as an extra channel or after flattening)
2. The latent code z before decoding

This allows us to say "generate me a 7" and get a 7.

In [ ]:
NUM_CLASSES = 10
CVAE_LATENT = 16

class CVAEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Image encoder
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(),  # 28→14
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), # 14→7
        )
        # After flattening (64*7*7=3136) + one-hot (10) → latent
        self.fc_mu = nn.Linear(64*7*7 + NUM_CLASSES, CVAE_LATENT)
        self.fc_logvar = nn.Linear(64*7*7 + NUM_CLASSES, CVAE_LATENT)

    def forward(self, x, c):
        h = self.conv(x).view(x.size(0), -1)
        h_c = torch.cat([h, c], dim=1)  # concat class label
        return self.fc_mu(h_c), self.fc_logvar(h_c)


class CVAEDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        # z (16) + one-hot (10) → image
        self.fc = nn.Linear(CVAE_LATENT + NUM_CLASSES, 64*7*7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh(),
        )

    def forward(self, z, c):
        z_c = torch.cat([z, c], dim=1)
        h = self.fc(z_c).view(z.size(0), 64, 7, 7)
        return self.deconv(h)


class CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CVAEEncoder()
        self.decoder = CVAEDecoder()

    def reparameterize(self, mu, logvar):
        if self.training:
            return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return mu

    def forward(self, x, label):
        c = F.one_hot(label, num_classes=NUM_CLASSES).float()
        mu, logvar = self.encoder(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z, c), mu, logvar

# Train CVAE
torch.manual_seed(42)
cvae = CVAE().to(device)
cvae_optimizer = optim.Adam(cvae.parameters(), lr=1e-3)
CVAE_EPOCHS = 20

print('Training Conditional VAE for 20 epochs...')
for epoch in range(CVAE_EPOCHS):
    cvae.train()
    epoch_loss = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        cvae_optimizer.zero_grad()
        recon, mu, logvar = cvae(imgs, lbls)
        loss, _, _ = vae_loss(recon, imgs, mu, logvar)
        loss.backward()
        cvae_optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/{CVAE_EPOCHS} | Loss: {epoch_loss/len(train_loader):.2f}')

print('\nConditional VAE training complete!')

# Generate digits 0-9 by conditioning
cvae.eval()
with torch.no_grad():
    z_fixed = torch.randn(10, CVAE_LATENT, device=device)
    labels_0_9 = torch.arange(10, device=device)
    c_0_9 = F.one_hot(labels_0_9, num_classes=NUM_CLASSES).float()
    gen_digits = cvae.decoder(z_fixed, c_0_9).cpu()

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i in range(10):
    axes[i].imshow(unnorm(gen_digits[i]).squeeze(), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(str(i), fontsize=10)

plt.suptitle('Conditional VAE: Generated Digits 0-9 (same z, different class label)', fontsize=12)
plt.tight_layout()
plt.show()

### How to Read This Chart: Conditional VAE Generation

A row of 10 generated images, one for each digit class 0-9. The same latent vector z is used for all 10 — only the class label changes.

- **Each image** is labeled with the digit class it was conditioned on (0 through 9).
- **Same z, different class** — this demonstrates that the class label controls the identity of the generated digit, while z controls stylistic variation within that class.

**What to look for:**
- Does each image actually look like the labeled digit? (0 should look like 0, 7 should look like 7, etc.)
- Style consistency — since the same z is used, there may be a common "handwriting style" across all 10 digits.
- Quality — CVAE images should be as good as or slightly better than unconditional VAE, because the model has more information (the label).

> **Why is this useful?** Conditional generation allows targeted data augmentation — if your dataset has only 50 examples of digit 9, you can generate 1000 more synthetically. This is widely used in medical imaging where certain pathologies are rare.

---

## Section 15 — Simplified FID Score

In [ ]:
# Simplified FID using pixel-space features (not Inception)
# True FID uses InceptionV3 features; we use flattened pixel vectors as a proxy.

def compute_simplified_fid(real_imgs_np, gen_imgs_np):
    """
    Simplified FID using flattened pixel features.
    True FID uses InceptionV3 features; this is a faithful structural proxy.
    Args:
        real_imgs_np: (N, H*W) flattened real images
        gen_imgs_np:  (N, H*W) flattened generated images
    """
    mu_r = np.mean(real_imgs_np, axis=0)
    mu_g = np.mean(gen_imgs_np, axis=0)
    sigma_r = np.cov(real_imgs_np, rowvar=False) + np.eye(real_imgs_np.shape[1]) * 1e-6
    sigma_g = np.cov(gen_imgs_np, rowvar=False) + np.eye(gen_imgs_np.shape[1]) * 1e-6
    diff = mu_r - mu_g
    # Matrix sqrt via eigendecomposition
    covmean, _ = linalg.sqrtm(sigma_r @ sigma_g, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff @ diff + np.trace(sigma_r + sigma_g - 2 * covmean)
    return float(fid)

N_FID = 1000

# Collect real images
real_imgs_list = []
for imgs, _ in test_loader:
    real_imgs_list.append(unnorm(imgs).view(imgs.size(0), -1).numpy())
    if sum(a.shape[0] for a in real_imgs_list) >= N_FID:
        break
real_feat = np.concatenate(real_imgs_list, axis=0)[:N_FID]

# VAE generated features
vae.eval()
with torch.no_grad():
    z_fid = torch.randn(N_FID, LATENT_DIM, device=device)
    vae_gen = vae.decoder(z_fid).cpu()
vae_feat = unnorm(vae_gen).view(N_FID, -1).numpy()

# DCGAN generated features
G.eval()
with torch.no_grad():
    z_fid2 = torch.randn(N_FID, Z_DIM, device=device)
    dcgan_gen = G(z_fid2).cpu()
dcgan_feat = unnorm(dcgan_gen).view(N_FID, -1).numpy()

print('Computing simplified FID scores (using pixel-space features)...')
print('(Note: True FID uses InceptionV3 features; this is a structural proxy)')

fid_vae = compute_simplified_fid(real_feat, vae_feat)
fid_dcgan = compute_simplified_fid(real_feat, dcgan_feat)

print('\n=== Simplified FID Scores ===')
print(f'  Model       | FID Score')
print(f'  ------------|----------')
print(f'  VAE         | {fid_vae:.2f}')
print(f'  DCGAN       | {fid_dcgan:.2f}')
print(f'\nLower FID = closer to real data distribution.')

### How to Read This Output: FID Scores

The **Fréchet Inception Distance (FID)** is the standard metric for evaluating generative image models.

**What it measures:**
- FID computes the distance between the distribution of real images and the distribution of generated images in feature space.
- True FID uses features from a pretrained InceptionV3 network. Our simplified version uses raw pixel features as a structural proxy.
- **Lower FID = better** — generated images are more similar to real images.

**Interpretation:**
- FID = 0 would mean generated images are statistically identical to real images (impossible in practice).
- DCGAN typically achieves lower FID than VAE because it generates sharper images.
- VAE images are often blurrier (higher MSE error), which the FID penalizes.

**Limitations of simplified FID:**
- We use pixel features (784 dimensions) instead of Inception features (~2048 dimensions).
- Pixel FID doesn't capture high-level perceptual quality well.
- Always report whether you used true FID (Inception features) or a proxy.

> **Reference values:** On MNIST, a well-trained DCGAN achieves a true FID of ~2-10. A VAE typically achieves ~10-30. A random noise baseline scores ~300+.

---

## Section 16 — VAE vs GAN Comparison Table

| Property | VAE | DCGAN | WGAN |
|----------|-----|-------|------|
| **Training stability** | Very stable | Unstable (mode collapse, oscillation) | More stable than DCGAN |
| **Image quality** | Good (slightly blurry) | Sharp, high quality | Similar to DCGAN |
| **Latent space** | Structured, continuous, interpolatable | Unstructured | Unstructured |
| **Density estimation** | Yes (ELBO approximates log p(x)) | No | No |
| **Conditional generation** | Easy (CVAE) | Possible (CGAN) | Possible |
| **Mode coverage** | Good (all modes represented) | Can drop modes | Better than DCGAN |
| **Loss meaningful?** | Yes (ELBO correlates with quality) | No (BCE doesn't indicate quality) | Yes (Wasserstein ≈ quality) |
| **Best for** | Latent space exploration, anomaly detection, drug discovery | High-quality image synthesis | Stable GAN training |

### When to Use Each

**Use a VAE when:**
- You need an **interpretable, manipulable latent space** (interpolation, arithmetic)
- You need **anomaly detection** (high reconstruction error = anomaly)
- You need **density estimation** (probability of a new sample under the model)
- Training stability matters more than image sharpness

**Use a GAN when:**
- You need the **highest visual quality** (photorealistic images)
- You don't need to infer the latent code from new images
- You can tolerate training instability

**Use WGAN over DCGAN when:**
- Your DCGAN training is unstable
- You want a loss that correlates with generation quality
- You're training on small or imbalanced datasets

---

---

## Summary

### What We Built

In this notebook we built and trained five generative models from scratch on MNIST:

1. **VAE** — learned a structured 16-dimensional latent space, reconstructed digits, interpolated between digit classes
2. **VAE Latent Space Analysis** — PCA visualization showing class-separated clusters
3. **DCGAN** — adversarial training, mode collapse demo, training progression visualization
4. **WGAN** — Wasserstein distance, weight clipping, more stable training dynamics
5. **Conditional VAE** — class-conditioned generation, generating specific digits on demand

### Evaluation Methods

Evaluating generative models is genuinely hard — there's no single correct metric:

| Metric | What It Measures | Limitation |
|--------|-----------------|------------|
| FID | Distribution similarity (perceptual features) | Needs large sample size; Inception model bias |
| Inception Score (IS) | Quality and diversity | Doesn't compare to real data |
| SSIM | Structural similarity to real images | Doesn't capture semantic quality |
| Reconstruction MSE | VAE reconstruction accuracy | Penalizes blur, rewards sharpness |
| Human evaluation | Perceived realism | Expensive, subjective |

### A Note on Diffusion Models

Since 2020, **diffusion models** (DDPM, Score Matching, Stable Diffusion) have superseded GANs for most image generation tasks:

- **How they work:** Train a neural network to reverse a gradual Gaussian noise-adding process. Start from pure noise, denoise 1000 steps to get a clean image.
- **Why they're better:** More stable training, better mode coverage, higher quality than GANs.
- **Relationship to VAEs:** Diffusion models can be seen as hierarchical VAEs with a fixed encoder (the noising process).

The foundational concepts from this notebook — latent spaces, the reparameterization trick, adversarial training — are all present in modern diffusion models. VAEs and GANs are the essential prerequisites for understanding DALL-E, Stable Diffusion, and Midjourney.

### Common Mistakes to Avoid

1. **Not normalizing images to [-1, 1]** — Tanh output requires this range.
2. **Training G and D at the same learning rate** — D usually needs a lower LR or more steps.
3. **Forgetting label smoothing** — Using 1.0 labels for real is too strong; 0.9 helps.
4. **Too many D updates per G update** — D dominates, G gradient vanishes. 1:1 or 5:1 (for WGAN) works.
5. **No early stopping on FID** — GAN performance can degrade after a certain epoch. Monitor FID, not just loss.
6. **Ignoring posterior collapse in VAEs** — Monitor KL per epoch. If KL → 0, the decoder ignores z.

---

*This completes the Deep Learning Series: 03_sequence_models → 04_attention_transformers → 05_generative_models*